# Visualisasi Transaksi POS Tahun 2025

Empat grafik dibuat menggunakan Matplotlib dari dataset transaksi POS yang sama.

## Persiapan Data

Data dibersihkan dengan menghapus duplikat, merapikan format harga, dan mengubah tanggal ke format waktu. Analisis penjualan memakai transaksi selesai dengan jumlah 1–12 unit.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv("paket_a_transaksi_pos_2024_2025.csv", dtype=str)
data = data.drop_duplicates().copy()

data["harga_satuan"] = (
    data["harga_satuan"]
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

data["jumlah"] = data["jumlah"].astype(int)
data["diskon_persen"] = data["diskon_persen"].astype(int)
data["tanggal"] = pd.to_datetime(data["tanggal"], format="mixed", dayfirst=True)
data["tahun"] = data["tanggal"].dt.year

data_2025 = data[data["tahun"] == 2025].copy()
data_jual = data_2025[
    (data_2025["status"] == "SELESAI")
    & (data_2025["jumlah"].between(1, 12))
].copy()

data_jual["Penjualan_Bersih"] = (
    data_jual["harga_satuan"]
    * data_jual["jumlah"]
    * (1 - data_jual["diskon_persen"] / 100)
)

print("Transaksi tahun 2025:", len(data_2025))
print("Transaksi selesai yang dipakai untuk penjualan:", len(data_jual))

## 1. Penjualan per Bulan Tahun 2025

Penjualan bersih dihitung dari harga dikali jumlah, setelah dikurangi diskon. Pajak tidak dimasukkan ke penjualan.

In [ ]:
penjualan_bulanan = data_jual.groupby(
    data_jual["tanggal"].dt.month
)["Penjualan_Bersih"].sum()

nama_bulan = [
    "Jan", "Feb", "Mar", "Apr", "Mei", "Jun",
    "Jul", "Agu", "Sep", "Okt", "Nov", "Des"
]

penjualan_bulanan = penjualan_bulanan.reindex(range(1, 13), fill_value=0)

plt.figure(figsize=(10, 5))
plt.bar(nama_bulan, penjualan_bulanan.values, color="#3569A8")
plt.title("Penjualan Bersih per Bulan — 2025")
plt.xlabel("Bulan")
plt.ylabel("Penjualan Bersih (Rp)")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig("visualisasi_penjualan_bulanan_2025.png", dpi=160)
plt.show()

## 2. Sepuluh Produk Terlaris Tahun 2025

Produk diurutkan berdasarkan total unit terjual pada transaksi selesai. Dataset berisi kode produk, bukan nama produk.

In [ ]:
produk_terlaris = (
    data_jual.groupby("kode_produk")["jumlah"]
    .sum()
    .nlargest(10)
    .sort_values()
)

plt.figure(figsize=(9, 6))
plt.barh(produk_terlaris.index, produk_terlaris.values, color="#4D8C78")
plt.title("10 Produk Terlaris — 2025")
plt.xlabel("Total Unit Terjual")
plt.ylabel("Kode Produk")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.savefig("visualisasi_10_produk_terlaris_2025.png", dpi=160)
plt.show()

## 3. Proporsi Transaksi Berhasil dan Tidak Selesai

Status berhasil adalah `SELESAI`. Status `BATAL` dan `RETUR` digabung sebagai transaksi tidak selesai agar proporsi dapat dibandingkan dalam dua kelompok. Jumlah `BATAL` dan `RETUR` tetap ditampilkan terpisah.

In [ ]:
jumlah_status = data_2025["status"].value_counts()

berhasil = jumlah_status.get("SELESAI", 0)
tidak_selesai = jumlah_status.get("BATAL", 0) + jumlah_status.get("RETUR", 0)
total_status = berhasil + tidak_selesai

plt.figure(figsize=(8, 3.5))
plt.barh(["Transaksi"], [berhasil], color="#4D8C78", label="SELESAI")
plt.barh(
    ["Transaksi"],
    [tidak_selesai],
    left=[berhasil],
    color="#D49A48",
    label="BATAL + RETUR"
)
plt.xlim(0, total_status)
plt.title("Proporsi Status Transaksi — 2025")
plt.xlabel("Jumlah transaksi")
plt.yticks([])
plt.legend(loc="lower center", bbox_to_anchor=(0.5, -0.5), ncol=2, frameon=False)
plt.tight_layout()
plt.savefig("visualisasi_status_transaksi_2025.png", dpi=160)
plt.show()

print("SELESAI:", berhasil)
print("BATAL:", jumlah_status.get("BATAL", 0))
print("RETUR:", jumlah_status.get("RETUR", 0))

## 4. Hubungan Jumlah Pembelian dengan Diskon

Setiap titik menunjukkan kombinasi jumlah barang dan persentase diskon. Ukuran titik menunjukkan banyaknya transaksi pada kombinasi tersebut.

In [ ]:
hubungan = (
    data_jual.groupby(["jumlah", "diskon_persen"])
    .size()
    .reset_index(name="frekuensi")
)

plt.figure(figsize=(9, 5))
plt.scatter(
    hubungan["jumlah"],
    hubungan["diskon_persen"],
    s=hubungan["frekuensi"] * 0.75 + 25,
    alpha=0.6,
    color="#8A5AA6"
)
plt.title("Jumlah Pembelian dan Diskon — 2025")
plt.xlabel("Jumlah Barang Dibeli (unit)")
plt.ylabel("Diskon (%)")
plt.xticks(sorted(hubungan["jumlah"].unique()))
plt.yticks(sorted(hubungan["diskon_persen"].unique()))
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("visualisasi_jumlah_dan_diskon_2025.png", dpi=160)
plt.show()

## Catatan

Grafik penjualan dan produk memakai transaksi `SELESAI` dengan jumlah 1–12 unit. Batas ini digunakan agar nilai kuantitas yang tidak wajar tidak mendominasi hasil. Grafik status memakai semua transaksi tahun 2025 setelah duplikat dihapus. Hubungan jumlah dan diskon menunjukkan pola data, bukan sebab-akibat.